# Colab Submission Notebook

This is the Colab notebook we would submit with our Iberdrola Datathon project.

We designed it as the shortest evaluator-friendly path through the work: it sets up the repo in **Google Colab**, validates the required submission files already tracked in the repository, and previews the main deliverables.

## What this notebook does

1. Clones the repository into Colab
2. Installs the dependencies needed to run the project
3. Checks that the required CSV and HTML deliverables are present in the repo
4. Validates `File 1.csv`, `File 2.csv`, and `File 3.csv`
5. Previews the outputs and links to the final HTML artifacts

## Expected runtime

Around `5-10 minutes`, depending on Colab performance.


## 1. Colab Setup

Run the next cell first. It will:
- clone the repository if needed
- install the main Python dependencies
- switch the notebook into the project directory

If the repo is already present in the Colab runtime, the cell will reuse it.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/npbeard/datathon-iberdrola-gooners.git"
REPO_DIR = Path("/content/datathon-iberdrola-gooners")
REQUIRED_PATHS = [
    REPO_DIR / "scripts" / "run_pipeline.py",
    REPO_DIR / "scripts" / "generate_submission_package.py",
    REPO_DIR / "src" / "visualization" / "offline_dashboard.py",
]

repo_is_complete = REPO_DIR.exists() and all(path.exists() for path in REQUIRED_PATHS)

if not repo_is_complete:
    if REPO_DIR.exists():
        print(f"Found an incomplete repo copy at {REPO_DIR}, so we are replacing it.")
        subprocess.run(["rm", "-rf", str(REPO_DIR)], check=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repository already present at {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)

try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
except subprocess.CalledProcessError:
    print("\nThe direct requirements install failed in this Colab runtime, so we are trying a more Colab-friendly fallback install.\n")
    fallback_packages = [
        "requests>=2.31.0",
        "pandas>=2.2.0",
        "polars>=0.20.0",
        "pyarrow>=16.0.0",
        "shapely>=2.0.0",
        "pyproj",
        "fiona",
        "geopandas>=0.14.0",
        "pyyaml>=6.0",
        "scikit-learn>=1.3.0",
        "statsmodels>=0.14.0",
        "folium>=0.14.0",
        "plotly>=5.17.0",
        "ipython>=8.0.0",
        "notebook>=7.0.0",
        "jupyter>=1.0.0",
        "pytest>=7.0.0",
        "pytest-cov>=5.0.0",
        "requests-mock>=1.9.0",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", *fallback_packages], check=True)

print("Setup complete.")

## 2. Validate The Final Submission Package

This is the main execution step.

For the Colab submission, we keep the workflow focused on the final deliverables that are already tracked in the repository. The notebook checks that the required files are present and then runs the submission validator on the official CSV outputs.

This Colab version does **not** rerun the full preprocessing pipeline from raw data. The full pipeline code is still included in the repository, but this notebook is intentionally set up as the most reliable review path for the final submission assets.

It runs:
- a file-presence check for the main CSV and HTML deliverables
- the submission validator for the required CSV files


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
SUBMISSION_DIR = ROOT / "data" / "submission"
MAP_PATH = ROOT / "maps" / "proposed_charging_network.html"
DASHBOARD_PATH = ROOT / "maps" / "dashboard.html"
EXPLORER_PATH = ROOT / "maps" / "offline_scenario_explorer.html"
ASSUMPTIONS_PATH = SUBMISSION_DIR / "ASSUMPTIONS.md"

required_outputs = [
    SUBMISSION_DIR / "File 1.csv",
    SUBMISSION_DIR / "File 2.csv",
    SUBMISSION_DIR / "File 3.csv",
    ASSUMPTIONS_PATH,
    MAP_PATH,
    DASHBOARD_PATH,
    EXPLORER_PATH,
]
missing_outputs = [path for path in required_outputs if not path.exists()]
if missing_outputs:
    missing_text = "\n".join(str(path) for path in missing_outputs)
    raise FileNotFoundError(f"The cloned repository is missing required submission assets:\n{missing_text}")

def run_command(command, env):
    print(f"\n$ {' '.join(command)}")
    completed = subprocess.run(
        command,
        cwd=ROOT,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {' '.join(command)}")

env = os.environ.copy()
existing_pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = str(ROOT) if not existing_pythonpath else f"{ROOT}:{existing_pythonpath}"

print("All required submission assets are present in the cloned repository.")
run_command([sys.executable, str(ROOT / "scripts" / "validate_submission.py")], env)

print("\nSubmission package assets are present and the required CSV files validated successfully.")


## 3. Preview The Required CSV Deliverables

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

file_1 = pd.read_csv(SUBMISSION_DIR / "File 1.csv")
file_2 = pd.read_csv(SUBMISSION_DIR / "File 2.csv")
file_3 = pd.read_csv(SUBMISSION_DIR / "File 3.csv")

display(Markdown("### File 1.csv"))
display(file_1)

display(Markdown("### File 2.csv"))
display(file_2.head(10))
print(f"File 2 rows: {len(file_2)}")

display(Markdown("### File 3.csv"))
display(file_3.head(10))
print(f"File 3 rows: {len(file_3)}")

## 4. Show Assumptions And Submission Notes

In [ ]:
from IPython.display import Markdown, display

assumptions_path = SUBMISSION_DIR / "ASSUMPTIONS.md"
summary_path = ROOT / "docs" / "executive_summary.md"

display(Markdown("### Assumptions"))
display(Markdown(assumptions_path.read_text(encoding="utf-8")))

display(Markdown("### Executive Summary"))
display(Markdown(summary_path.read_text(encoding="utf-8")))

## 5. Access The Final HTML Artifacts

The HTML artifacts are generated inside the Colab runtime. The next cell prints their paths and creates clickable links.

In [ ]:
from IPython.display import HTML, display

display(HTML(f"<p><a href='{MAP_PATH.as_posix()}' target='_blank'>Open proposed charging network map</a></p>"))
display(HTML(f"<p><a href='{DASHBOARD_PATH.as_posix()}' target='_blank'>Open full dashboard</a></p>"))
display(HTML(f"<p><a href='{EXPLORER_PATH.as_posix()}' target='_blank'>Open offline scenario explorer</a></p>"))

print(MAP_PATH)
print(DASHBOARD_PATH)
print(EXPLORER_PATH)

## 6. Optional: Download The Submission Package From Colab

Run the next cell if you want to bundle the required CSV files and assumptions note into a single zip file from Colab.

In [ ]:
import shutil
from pathlib import Path

zip_base = ROOT / "data" / "submission"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=SUBMISSION_DIR)
print(f"Created zip archive: {zip_path}")